# FLUX.1 LoRA Training — Free Colab Profile

This notebook is tuned for **free-tier Colab** (~12 GB CPU RAM, ~15 GB GPU VRAM).

**Profile used here:**
- Base model: `FLUX.1-schnell` (not FLUX.1-dev)
- Resolution: `512`
- Batch size: `1`, LoRA rank: `4`
- Memory flags: gradient checkpointing, fp16, 8-bit Adam, latent caching

**Before you start**
1. Runtime → Change runtime type → **GPU** (T4 is fine)
2. Accept the FLUX license on Hugging Face: https://huggingface.co/black-forest-labs/FLUX.1-schnell
3. Create a Hugging Face access token: https://huggingface.co/settings/tokens
4. Put your raw training images in Google Drive (see step 4)

Expected output: `training/output/mystyle_flux_lora.safetensors`

In [ ]:
# Check GPU memory (expect ~15 GB on free T4)
!nvidia-smi

In [ ]:
# Set your repo URL once (replace with your GitHub repo)
REPO_URL = "https://github.com/PariGarg10/ARTGENERATOR.git"
BRANCH = "main"

# Optional: folder on Google Drive with raw images (0000.png, 0001.png, ...)
DRIVE_RAW_IMAGES = "/content/drive/MyDrive/flux_dataset/raw"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo and auto-detect the folder with training scripts
import os
from pathlib import Path

%cd /content
!rm -rf project artgenerator

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError(
        "Set REPO_URL in the cell above to your real GitHub repo, e.g.\n"
        'REPO_URL = "https://github.com/parigarg/artgenerator.git"'
    )

!git clone --branch {BRANCH} {REPO_URL} project

clone_root = Path("/content/project")
candidates = [
    clone_root / "project",  # repo layout: artgenerator/project/...
    clone_root,              # repo layout: scripts at repo root
]

work_dir = None
for path in candidates:
    if (path / "training" / "train_flux_lora.py").exists():
        work_dir = path
        break

if work_dir is None:
    print("Clone root contents:")
    !ls -la /content/project
    raise FileNotFoundError(
        "Could not find training/train_flux_lora.py. "
        "Check REPO_URL, push your latest code, and re-run this cell."
    )

os.chdir(work_dir)
print("Working directory:", work_dir)
!pwd
!ls

In [ ]:
# Install Colab-tested library versions (must match bundled training script v0.34.0)
!pip uninstall -y diffusers -q
!pip install -q -r requirements-colab.txt
!pip install -q "peft>=0.15.0"

import diffusers
import peft
from diffusers.training_utils import _collate_lora_metadata
print("diffusers:", diffusers.__version__, "| peft:", peft.__version__)

# Verify the bundled HF script (do NOT delete it)
from pathlib import Path
script = Path("training/_hf_scripts/train_dreambooth_lora_flux.py")
text = script.read_text(encoding="utf-8") if script.exists() else ""
if "check_min_version(\"0.34.0\")" not in text:
    raise RuntimeError(
        "Wrong training script. Run: git pull, or re-clone the repo. "
        f"Found script size={script.stat().st_size if script.exists() else 0} bytes."
    )
print("Training script OK (diffusers 0.34.0 compatible)")

In [ ]:
# Hugging Face login (REQUIRED before training — run this cell every new Colab session)
from google.colab import userdata
from huggingface_hub import login, whoami
from transformers import CLIPTokenizer

MODEL_ID = "black-forest-labs/FLUX.1-schnell"

# Option A: Colab secret named HF_TOKEN (recommended)
#   Left panel -> Secrets -> Add HF_TOKEN = your huggingface.co/settings/tokens value
try:
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("Logged in via HF_TOKEN secret")
except Exception:
    # Option B: paste token when prompted
    login()
    print("Logged in via interactive token")

print("HF user:", whoami()["name"])

# Must succeed before training starts
CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
print("Tokenizer OK — you can run training")

## 1) Copy dataset into the notebook

Upload images to Google Drive at the path you set in `DRIVE_RAW_IMAGES`, **or** upload directly into `dataset/raw` in the next cell.

In [ ]:
!mkdir -p dataset/raw dataset/processed

import os
import shutil
from pathlib import Path

raw_dir = Path("dataset/raw")
drive_raw = Path(DRIVE_RAW_IMAGES)

if drive_raw.exists():
    for item in drive_raw.iterdir():
        if item.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp"}:
            shutil.copy2(item, raw_dir / item.name)
    print(f"Copied images from Drive: {drive_raw}")
else:
    print(f"Drive folder not found: {drive_raw}")
    print("Upload images manually to dataset/raw, then re-run this cell.")

image_count = len(list(raw_dir.glob("*")))
print(f"Raw images ready: {image_count}")

In [ ]:
# Resize/clean images to 512x512 for free-tier VRAM
!python dataset/preprocess_dataset.py \
  --input_dir ./dataset/raw \
  --output_dir ./dataset/processed \
  --size 512

In [ ]:
# Auto-caption on Colab (Florence-2 with BLIP fallback)
!python captions/generate_florence_captions.py \
  --image_dir ./dataset/processed \
  --trigger_token mystyle

## 2) Train LoRA (free Colab profile)

This uses `FLUX.1-schnell`, rank 4, 512px, and memory-saving flags.
Training may take 30–90 minutes depending on dataset size and GPU queue.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
print("GPU cleared before training.")

In [ ]:
# Quick sanity check before training (should print diffusers 0.34.x)
import diffusers
from diffusers.training_utils import _collate_lora_metadata
print("OK: diffusers", diffusers.__version__, "| _collate_lora_metadata available")

!python training/train_flux_lora.py \
  --dataset_dir ./dataset/processed \
  --output_dir ./training/output \
  --trigger_token mystyle \
  --profile free_colab

In [ ]:
# Verify output
!ls -lah ./training/output

In [ ]:
# Copy trained LoRA back to Google Drive
from pathlib import Path
import shutil

lora_file = Path("training/output/mystyle_flux_lora.safetensors")
drive_out = Path("/content/drive/MyDrive/flux_training_output")
drive_out.mkdir(parents=True, exist_ok=True)

if lora_file.exists():
    dest = drive_out / lora_file.name
    shutil.copy2(lora_file, dest)
    print(f"Saved to Drive: {dest}")
else:
    print("LoRA file not found. Check training logs above.")

## 3) After training (local machine)

Download `mystyle_flux_lora.safetensors` from Drive, place it in `training/output/`, then run:

```bash
python inference/generate.py \
  --prompt "a cozy cafe in rain" \
  --lora_path ./training/output/mystyle_flux_lora.safetensors \
  --base_model black-forest-labs/FLUX.1-schnell \
  --width 512 --height 512
```

Or launch the Gradio app:

```bash
python gradio_app.py
```

---

### Optional: high-VRAM profile (A100 / 24GB+)

Only if you have a paid Colab Pro GPU with enough VRAM:

```bash
python training/train_flux_lora.py \
  --dataset_dir ./dataset/processed \
  --output_dir ./training/output \
  --trigger_token mystyle \
  --profile high_vram
```